<a href="https://colab.research.google.com/github/icosahedron31/Walmart-Sales/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wandb onnx -Uq

In [2]:
!kaggle --version

Kaggle API 1.6.17


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [5]:
!cp /content/drive/MyDrive/ColabNotebooks/kaggle_API_credentials/kaggle.json ~/.kaggle/kaggle.json

In [6]:
! chmod 600 ~/.kaggle/kaggle.json

In [7]:
!pip uninstall -y kaggle kagglesdk
!pip install --force-reinstall --no-cache-dir "kaggle==1.6.17"

Found existing installation: kaggle 2.0.2
Uninstalling kaggle-2.0.2:
  Successfully uninstalled kaggle-2.0.2
Found existing installation: kagglesdk 0.1.23
Uninstalling kagglesdk-0.1.23:
  Successfully uninstalled kagglesdk-0.1.23
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 247.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 235.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 205.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 212.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 245.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 269.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 154.6 MB/s eta 0:00:00
   

In [7]:
!kaggle --version

Kaggle API 1.6.17


In [8]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting

In [9]:
! unzip walmart-recruiting-store-sales-forecasting

Archive:  walmart-recruiting-store-sales-forecasting.zip
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [10]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip


Archive:  features.csv.zip
  inflating: features.csv            
Archive:  test.csv.zip
  inflating: test.csv                
Archive:  train.csv.zip
  inflating: train.csv               


In [11]:
! unzip sampleSubmission.csv.zip

Archive:  sampleSubmission.csv.zip
  inflating: sampleSubmission.csv    


**Imports**

In [12]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cpu


**Reading Data**

In [13]:
df_sales = pd.read_csv("train.csv")
df_stores = pd.read_csv("stores.csv")
df_features = pd.read_csv("features.csv")

In [20]:
df_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB


In [18]:
df_stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [19]:
df_stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [ ]:
df_sales.describe()

,Store,Dept,Weekly_Sales
count,421570.000000,421570.000000,421570.000000
mean,22.200546,44.260317,15981.258123
std,12.785297,30.492054,22711.183519
min,1.000000,1.000000,-4988.940000
25%,11.000000,18.000000,2079.650000
50%,22.000000,37.000000,7612.030000
75%,33.000000,74.000000,20205.852500
max,45.000000,99.000000,693099.360000


In [23]:
df_features["Fuel_Price"].describe()

,Fuel_Price
count,8190.000000
mean,3.405992
std,0.431337
min,2.472000
25%,3.041000
50%,3.513000
75%,3.743000
max,4.468000


In [ ]:
df_features.describe()

,Store,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
count,8190.000000,8190.000000,8190.000000,4032.000000,2921.000000,3613.000000,3464.000000,4050.000000,7605.000000,7605.000000
mean,23.000000,59.356198,3.405992,7032.371786,3384.176594,1760.100180,3292.935886,4132.216422,172.460809,7.826821
std,12.987966,18.678607,0.431337,9262.747448,8793.583016,11276.462208,6792.329861,13086.690278,39.738346,1.877259
min,1.000000,-7.290000,2.472000,-2781.450000,-265.760000,-179.260000,0.220000,-185.170000,126.064000,3.684000
25%,12.000000,45.902500,3.041000,1577.532500,68.880000,6.600000,304.687500,1440.827500,132.364839,6.634000
50%,23.000000,60.710000,3.513000,4743.580000,364.570000,36.260000,1176.425000,2727.135000,182.764003,7.806000
75%,34.000000,73.880000,3.743000,8923.310000,2153.350000,163.150000,3310.007500,4832.555000,213.932412,8.567000
max,45.000000,101.950000,4.468000,103184.980000,104519.540000,149483.310000,67474.850000,771448.100000,228.976456,14.313000


In [ ]:
df_test = pd.read_csv("test.csv")
df_test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [27]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class EconomicIndicatorImputer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=("CPI", "Unemployment")):
        self.columns = list(columns)

    def fit(self, X, y=None):
        X = X.copy()
        X = X.sort_values(["Store", "Date"])

        self.last_values_ = (
            X.groupby("Store")[self.columns]
             .last()
        )

        self.global_values_ = X[self.columns].mean()

        return self

    def transform(self, X):
        X = X.copy()
        X = X.sort_values(["Store", "Date"])

        for col in self.columns:
            # Fill gaps within each store
            X[col] = (
                X.groupby("Store")[col]
                 .transform(lambda s: s.ffill())
            )

            # Fill remaining NaNs using values learned during fit()
            missing = X[col].isna()

            X.loc[missing, col] = (
                X.loc[missing, "Store"]
                 .map(self.last_values_[col])
            )

            # Final fallback for unseen stores
            X[col] = X[col].fillna(self.global_values_[col])

        return X

In [57]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class MarkdownImputer(BaseEstimator, TransformerMixin):
    """
    Fills missing markdown/promotion columns with a constant (default 0).
    NaN in these columns typically means "no markdown was running".
    """

    def __init__(
        self,
        columns=("MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"),
        fill_value=0,
    ):
        self.columns = list(columns)
        self.fill_value = fill_value

    def fit(self, X, y=None):
        # No stats need to be learned, but sklearn's fitted-check
        # looks for a trailing-underscore attribute — this satisfies it.
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            X[col] = X[col].fillna(self.fill_value)
        return X

In [48]:
# Check for duplicate keys before merging
print("Duplicate Store rows in stores.csv:", df_stores.duplicated(subset=["Store"]).sum())
print("Duplicate Store/Date rows in features.csv:", df_features.duplicated(subset=["Store", "Date"]).sum())
print("Duplicate Store/Date rows in train_raw:", df_sales.duplicated(subset=["Store", "Date"]).sum())
print(df_sales.duplicated(subset=["Store", "Dept", "Date"]).sum())  # should be 0 or near-0


Duplicate Store rows in stores.csv: 0
Duplicate Store/Date rows in features.csv: 0
Duplicate Store/Date rows in train_raw: 415135
0


In [58]:
train_full = df_sales.merge(df_features, on=["Store", "Date"], how="left")
train_full = train_full.merge(df_stores, on="Store", how="left")

# Make sure Date is an actual datetime, not a string
train_full["Date"] = pd.to_datetime(train_full["Date"])


In [59]:
train_full = train_full.sort_values(["Store", "Date"]).reset_index(drop=True)

# e.g. last 20% of the date range becomes validation
cutoff_date = train_full["Date"].quantile(0.8)

X_train = train_full[train_full["Date"] <= cutoff_date].copy()
X_val   = train_full[train_full["Date"] > cutoff_date].copy()

In [60]:
X_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,2,2010-02-05,50605.27,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
2,1,3,2010-02-05,13740.12,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
3,1,4,2010-02-05,39954.04,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
4,1,5,2010-02-05,32229.38,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315


In [61]:
X_val.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
8245,1,1,2012-04-20,16976.19,False,66.76,3.877,2230.8,612.02,19.75,275.13,5747.1,221.564074,7.143,False,A,151315
8246,1,2,2012-04-20,45561.85,False,66.76,3.877,2230.8,612.02,19.75,275.13,5747.1,221.564074,7.143,False,A,151315
8247,1,3,2012-04-20,8647.36,False,66.76,3.877,2230.8,612.02,19.75,275.13,5747.1,221.564074,7.143,False,A,151315
8248,1,4,2012-04-20,36350.03,False,66.76,3.877,2230.8,612.02,19.75,275.13,5747.1,221.564074,7.143,False,A,151315
8249,1,5,2012-04-20,16645.01,False,66.76,3.877,2230.8,612.02,19.75,275.13,5747.1,221.564074,7.143,False,A,151315


In [62]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([

    ("economic_indicator_imputer", EconomicIndicatorImputer()),
    ("markdown_imputer", MarkdownImputer(fill_value=0))
])

X_train_clean = pipeline.fit_transform(X_train)


In [63]:
X_train_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 338738 entries, 0 to 419685
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         338738 non-null  int64         
 1   Dept          338738 non-null  int64         
 2   Date          338738 non-null  datetime64[ns]
 3   Weekly_Sales  338738 non-null  float64       
 4   IsHoliday_x   338738 non-null  bool          
 5   Temperature   338738 non-null  float64       
 6   Fuel_Price    338738 non-null  float64       
 7   MarkDown1     338738 non-null  float64       
 8   MarkDown2     338738 non-null  float64       
 9   MarkDown3     338738 non-null  float64       
 10  MarkDown4     338738 non-null  float64       
 11  MarkDown5     338738 non-null  float64       
 12  CPI           338738 non-null  float64       
 13  Unemployment  338738 non-null  float64       
 14  IsHoliday_y   338738 non-null  bool          
 15  Type          338738 n

In [64]:
X_test_t = pipeline.transform(X_val).copy()